# Networks

You can start this section by uploading previously zip archive in /content

## Preparation

### Data

In [ ]:
# extract .zip
import zipfile
import shutil # deletes /content/content directory
import glob # searches for .zip files

try:
  shutil.rmtree("/content/saved_knots")
except Exception as e:
  pass

# get all .zip files from local storage
files = glob.glob('*.zip')

# if 2 or more files occured, we don't know which we want to extract
if len(files) > 1:
  print("Don't know which archive to unpack")
else:
  # unpack .zip
  with zipfile.ZipFile(f"/content/{files[0]}","r") as zip_ref:
      zip_ref.extractall("/content")

  # structure of .zip archive (for now) is archive.zip/content/saved_knots/, so after unpacking it we get /content/content/saved_knots dir, which is not comfortable to work with,
  shutil.move("/content/content/saved_knots", "/content/saved_knots") # so we move it 'higher' one level
  shutil.rmtree("/content/content") # and delete previous one

IndexError: list index out of range

In [ ]:
import torch
from PIL import Image
import tqdm
from tqdm.notebook import tqdm as tqdmn

In [ ]:
from torchvision.datasets import VisionDataset
import os
import os.path
import sys

IMG_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.ppm', '.bmp', '.pgm', '.tif', '.tiff', '.webp')

def has_file_allowed_extension(filename, extensions):
  return filename.lower().endswith(extensions)


def is_image_file(filename):
  return has_file_allowed_extension(filename, IMG_EXTENSIONS)


def make_dataset(dir, class_to_idx, extensions=None, is_valid_file=None):
  images = []
  dir = os.path.expanduser(dir)
  if not ((extensions is None) ^ (is_valid_file is None)):
    raise ValueError("Both extensions and is_valid_file cannot be None or not None at the same time")
  if extensions is not None:
    def is_valid_file(x):
      return has_file_allowed_extension(x, extensions)
  for target in sorted(class_to_idx.keys()):
    d = os.path.join(dir, target)
    if not os.path.isdir(d):
      continue
    for root, _, fnames in sorted(os.walk(d)):
      for fname in sorted(fnames):
        path = os.path.join(root, fname)
        if is_valid_file(path):
          item = (path, class_to_idx[target])
          images.append(item)

  return images

def pil_loader(path):
  with open(path, 'rb') as f:
    img = Image.open(f)
    return img.convert('RGB')

class CustomDatasetFolder(VisionDataset):
  def __init__(self, root, loader, extensions=None, transform=None,
               target_transform=None, is_valid_file=None):
    super().__init__(root, transform=transform,
                     target_transform=target_transform)
    classes, class_to_idx = self._find_classes(self.root)
    samples = make_dataset(self.root, class_to_idx, extensions, is_valid_file)
    if len(samples) == 0:
        raise (RuntimeError("Found 0 files in subfolders of: " + self.root + "\n"
                            "Supported extensions are: " + ",".join(extensions)))

    self.loader = loader
    self.extensions = extensions

    self.classes = classes
    self.class_to_idx = class_to_idx
    self.samples = samples
    self.targets = [s[1] for s in samples]

  def _find_classes(self, dir):
    if sys.version_info >= (3, 5):
        # Faster and available in Python 3.5 and above
        classes = [d.name for d in os.scandir(dir) if d.is_dir()]
    else:
        classes = [d for d in os.listdir(dir) if os.path.isdir(os.path.join(dir, d))]
    classes.sort()

    classes = [int(i) for i in classes]
    classes = [str(i) for i in range(0, max(classes)+1)]

    class_to_idx = {classes[i]: i for i in range(len(classes))}
    return classes, class_to_idx

  def __getitem__(self, index):
    path, target = self.samples[index]
    sample = self.loader(path)
    if self.transform is not None:
        sample = self.transform(sample)
    if self.target_transform is not None:
        target = self.target_transform(target)

    return sample, target

  def __len__(self):
    return len(self.samples)

class CustomImageFolder(CustomDatasetFolder):
  def __init__(self, root, transform=None, target_transform=None,
               loader=pil_loader, is_valid_file=None):
    super().__init__(root, loader, IMG_EXTENSIONS if is_valid_file is None else None,
                                          transform=transform,
                                          target_transform=target_transform,
                                          is_valid_file=is_valid_file)
    self.imgs = self.samples

In [ ]:
# we use torchvision.datasets.ImageFolder class to divide our data into classes and work with them efficiently
# useful links: https://pytorch.org/vision/main/generated/torchvision.datasets.ImageFolder.html
#
#               https://debuggercafe.com/pytorch-imagefolder-for-training-cnn-models/

import torchvision
from torchvision.transforms import v2
data_transform = v2.Compose([v2.Grayscale(num_output_channels=1),
                             v2.ToImage(),
                             v2.ToDtype(torch.float32, scale=True),
                             v2.functional.invert])

data = CustomImageFolder(root='/content/saved_knots',
                         transform=data_transform)

FileNotFoundError: [Errno 2] No such file or directory: '/content/saved_knots'

In [ ]:
data.classes

In [ ]:
type(data)

In [ ]:
# shape of the element of the data array
image = data[0][0]
image.shape

In [ ]:
IMG_SIZE = image.shape[1]

IMG_SIZE

In [ ]:
# split data into train and test sets
k = 0.8 # train/all ratio
train_len = int(k*len(data))
train_data, test_data = torch.utils.data.random_split(data, [train_len, len(data)-train_len])

In [ ]:
from torch.utils.data import DataLoader # iterable class; https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader

# Setup the batch size hyperparameter
BATCH_SIZE = 64

# Turn datasets into iterables (batches)
train_dataloader = DataLoader(train_data, # dataset to turn into iterable
    batch_size=BATCH_SIZE, # how many samples per batch?
    shuffle=True, # shuffle data every epoch?
    num_workers=2,
    pin_memory=True
)

test_dataloader = DataLoader(test_data,
                             num_workers=2,
                             pin_memory=True)

# Let's check out what we've created
print(f"Dataloaders: {train_dataloader, test_dataloader}")
print(f"Length of train dataloader: {len(train_dataloader)} batches of {BATCH_SIZE}")
print(f"Length of test dataloader: {len(test_dataloader)}")

In [ ]:
# Check out what's inside the training dataloader
train_features_batch, train_labels_batch = next(iter(train_dataloader))
train_features_batch.shape, train_labels_batch.shape

In [ ]:
train_data[0]

In [ ]:
# show a sample
import matplotlib.pyplot as plt

random_idx = torch.randint(0, len(train_features_batch), size=[1]).item()
img, label = train_features_batch[random_idx], train_labels_batch[random_idx]
plt.imshow(img.squeeze(), cmap="gray")
plt.title(data.classes[label])
plt.axis("Off");
print(f"Image size: {img.shape}")
print(f"Label: {label}, label size: {label.shape}")
print(data.classes)

In [ ]:
# device-agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu" # cuda is MUCH faster
device

In [ ]:
import requests
from pathlib import Path

# Download helper functions from Learn PyTorch repo (if not already downloaded)
if Path("helper_functions.py").is_file():
  print("helper_functions.py already exists, skipping download")
else:
  print("Downloading helper_functions.py")
  request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
  with open("helper_functions.py", "wb") as f:
    f.write(request.content)

### Functions for models

In [ ]:
# aka training loop
def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               accuracy_fn,
               device: torch.device = device):
    train_loss, train_acc = 0, 0
    y_pred_train, y_target_train = [], []
    for (X, y) in data_loader:
        # Send data to GPU
        X, y = X.to(device), y.to(device)

        # 1. Forward pass
        y_pred = model(X).squeeze(dim=1)

        for i in y_pred.tolist():
          y_pred_train.append(round(i))
        for i in y.tolist():
          y_target_train.append(round(i))

        # 2. Calculate loss
        loss = loss_fn(y_pred, y.type(torch.float32))
        train_loss += loss
        train_acc += accuracy_fn(y_true=y,
                                 y_pred=y_pred.round()) # Go from logits -> pred labels

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

    # Calculate loss and accuracy per epoch and print out what's happening
    train_loss /= len(data_loader)
    train_acc /= len(data_loader)
    print(f"\nTrain loss: {train_loss:.5f} | Train accuracy: {train_acc:.2f}%")
    return y_pred_train, y_target_train, train_loss.cpu().detach().numpy(), train_acc

# aka testing loop
def test_step(data_loader: torch.utils.data.DataLoader,
              model: torch.nn.Module,
              loss_fn: torch.nn.Module,
              accuracy_fn,
              threshold: float = 0.001,
              device: torch.device = device,
              scheduler: torch.optim.lr_scheduler = None,
              save_path: str = None):
    test_loss, test_acc = 0, 0
    y_pred_test, y_target_test = [], []
    model.eval() # put model in eval mode
    # Turn on inference context manager
    with torch.inference_mode():
        for X, y in data_loader:
            # Send data to GPU
            X, y = X.to(device), y.to(device)

            # 1. Forward pass
            test_pred = model(X).squeeze(dim=1)

            for i in test_pred.tolist():
              y_pred_test.append(round(i))
            for i in y.tolist():
              y_target_test.append(round(i))

            # 2. Calculate loss and accuracy
            test_loss += loss_fn(test_pred, y.type(torch.float32))
            test_acc += accuracy_fn(y_true=y,
                y_pred=test_pred.round() # Go from logits -> pred labels
            )

        # Adjust metrics and print out
        test_loss /= len(data_loader)
        test_acc /= len(data_loader)

        if scheduler != None:
          scheduler.step(test_loss)

        if test_acc > best_acc:
          best_fold = fold
          if save_path != None:
            torch.save(model.state_dict(), save_path + f'{model.__class__.__name__}_best.pth')

        print(f"\nTest loss: {test_loss:.5f} | Test accuracy: {test_acc:.2f}%\n")
        return y_pred_test, y_target_test, test_loss.cpu().detach().numpy(), test_acc

In [ ]:
def eval_model(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               accuracy_fn,
               device: torch.device = device):
    """Evaluates a given model on a given dataset.

    Args:
        model (torch.nn.Module): A PyTorch model capable of making predictions on data_loader.
        data_loader (torch.utils.data.DataLoader): The target dataset to predict on.
        loss_fn (torch.nn.Module): The loss function of model.
        accuracy_fn: An accuracy function to compare the models predictions to the truth labels.
        device (str, optional): Target device to compute on. Defaults to device.

    Returns:
        (dict): Results of model making predictions on data_loader.
    """
    loss, acc = 0, 0
    model.eval()
    with torch.inference_mode():
        for X, y in data_loader:
            # Send data to the target device
            X, y = X.to(device), y.to(device)
            y_pred = model(X).squeeze(dim=1)
            loss += loss_fn(y_pred, y)
            acc += accuracy_fn(y_true=y,
                                y_pred=y_pred.round())

        # Scale loss and acc
        loss /= len(data_loader)
        acc /= len(data_loader)
    return {"model_name": model.__class__.__name__, # only works when model was created with a class
            "model_loss": loss.item(),
            "model_acc": acc}

### Functions for plotting

In [ ]:
te = torch.Tensor([[1., 2., 3., 4., 5., 6., 7., 8., 9.],
                   [1., 2., 3., 4., 5., 6., 7., 8., 9.],
                   [1., 2., 3., 4., 5., 6., 7., 8., 9.]])

te = torch.sum(te, dim=0)

siz_te = int(te.shape[0]**(1/2))

te = torch.unflatten(te, dim=0, sizes=(siz_te, siz_te))

te, siz_te

In [ ]:
def plot_weights(layers: list):
  fig, axs = plt.subplots(ncols=1, nrows=len(layers), figsize=(7, 7*len(layers)))
  for i, layer in enumerate(layers):
    if type(layer) == torch.nn.modules.linear.Linear:
      weight = torch.sum(layer.weight, dim=0)
      siz = int(weight.shape[0]**(1/2))
      weight = torch.unflatten(weight, dim=0, sizes=(siz, siz))
    elif type(layer) == torch.nn.modules.conv.Conv2d:
      weight = torch.sum(torch.sum(layer.weight, dim=0), dim=0)
    axs[i].imshow(weight.cpu().detach().numpy())

In [ ]:
def plot_weights_image(layers: list, func_length: int, img: torch.Tensor):
  fig, axs = plt.subplots(ncols=2, nrows=func_length, figsize=(2*7, 7*func_length))
  i = -1
  for layer in layers:
    img = img.cpu()
    #print(type(layer), img.shape)
    if type(layer) == torch.nn.modules.linear.Linear:
      i += 1
      # how much does image activate each neuron
      img = img.squeeze()
      if torch.Tensor.dim(img) == 1:
        siz = int(img.shape[0]**(1/2))
        img = torch.unflatten(img, dim=0, sizes=(siz, siz))

      weight = torch.sum(layer.weight, dim=0).cpu()
      siz = int(weight.shape[0]**(1/2))
      weight = torch.unflatten(weight, dim=0, sizes=(siz, siz))
      mult = torch.mul(weight.cpu(), img.cpu()).cpu()
      axs[i][0].imshow(mult.detach().numpy())

      # output
      img = torch.flatten(img).unsqueeze(dim=0)
      img = layer(img.cuda()).cpu()

      siz = int(img.shape[1]**(1/2))
      img = torch.unflatten(img, dim=1, sizes=(siz, siz))

      axs[i][1].imshow(img.squeeze(dim=0).detach().numpy())

    elif type(layer) == torch.nn.modules.conv.Conv2d:
      i += 1
      weight = torch.sum(torch.sum(layer.weight, dim=0), dim=0)
      axs[i][0].imshow(weight.cpu().detach().numpy())

      img = layer(img.cuda()).cpu()

      img_show = torch.sum(img, dim=0)

      axs[i][1].imshow(img_show.squeeze(dim=0).detach().numpy())
    elif type(layer) == torch.nn.modules.flatten.Flatten:
      img = layer(img.unsqueeze(dim=0).cuda()).squeeze().cpu()
    elif type(layer) == torch.nn.modules.batchnorm.BatchNorm1d or type(layer) == torch.nn.modules.batchnorm.BatchNorm2d:
      pass
    else:
      img = layer(img.cuda()).cpu()

## CNN

### Building a model

In [ ]:
from torch import nn

# Create a convolutional neural network
class KnotsModelCNN(nn.Module):
    """
    Model architecture copying TinyVGG from:
    https://poloclub.github.io/cnn-explainer/

    To understand how it works, I highly recommend you go through the 'Convolutional Neural Networks' section at https://colah.github.io/ (of course, the other articles are very good too, so it's best to read them all!)
    """
    def __init__(self, output_shape: int):
        super().__init__()

        self.conv_1 = nn.Sequential(
          nn.Conv2d(1, 4, kernel_size=11, stride=1, dilation=2, padding=0),
          nn.BatchNorm2d(4),
          nn.ReLU(),

          nn.MaxPool2d(kernel_size=2),

          nn.Conv2d(4, 16, kernel_size=5, stride=1, dilation=2, padding=0),
          nn.BatchNorm2d(16),
          nn.ReLU(),

          nn.MaxPool2d(kernel_size=2),

          nn.Conv2d(16, 64, kernel_size=3, stride=1, dilation=2, padding=0),
          nn.BatchNorm2d(64),
          nn.ReLU(),

          nn.MaxPool2d(kernel_size=2),

          nn.Conv2d(64, 256, kernel_size=3, stride=1, dilation=2, padding=0),
          nn.BatchNorm2d(256),
          nn.ReLU(),

          nn.MaxPool2d(kernel_size=2),

          nn.Conv2d(256, 361, kernel_size=3, stride=1, padding=0),
          nn.BatchNorm2d(361),
          nn.ReLU(),

          nn.MaxPool2d(kernel_size=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(361*(12)**2),

            nn.Dropout(p=0.7),

            nn.Linear(in_features=361*(12)**2,
                      out_features=4096),
            nn.BatchNorm1d(4096),
            nn.ReLU(),

            nn.Dropout(p=0.7),

            nn.Linear(in_features=4096,
                      out_features=4096),
            nn.ReLU(),

            nn.Linear(4096, output_shape)
        )

    def forward(self, x: torch.Tensor):
      x = self.conv_1(x)
      #print(x.shape)
      x = self.classifier(x)
      return x

In [ ]:
model_2 = KnotsModelCNN(output_shape=1
                        ).to(device)

In [ ]:
def count_parameters(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)

count_parameters(model_2)

In [ ]:
param_size = 0
for param in model_2.parameters():
    param_size += param.nelement() * param.element_size()
buffer_size = 0
for buffer in model_2.buffers():
    buffer_size += buffer.nelement() * buffer.element_size()

size_all_mb = (param_size + buffer_size) / 1024**2
print('model size: {:.3f}MB'.format(size_all_mb))

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.AdamW(params=model_2.parameters(),
                              lr=0.0003,
                              weight_decay=0.0001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,
                                                        mode='min',
                                                        factor=0.2,
                                                        patience=2,
                                                        threshold=0.001)

In [ ]:
train_losses, train_accuracies = [], []
valid_losses, valid_accuracies = [], []

y_pred_train, y_target_train = [], []
y_pred_valid, y_target_valid = [], []

In [ ]:
# let's try our model
from helper_functions import accuracy_fn

EPOCHS = 30
best_acc = 0
best_fold = 0
fold = 0
SAVE_PATH = 'content/saved_models'

try:
  os.mkdir(SAVE_PATH)
except:
  pass


for epoch in tqdm.notebook.tqdm(range(EPOCHS)):
  print(f'\nEpoch: {epoch+1}')
  print(f'LR: {scheduler.get_last_lr()}')

  y_pred_train, y_target_train, train_loss, train_acc = train_step(data_loader=train_dataloader,
                                                                              model=model_2,
                                                                              loss_fn=loss_fn,
                                                                              optimizer=optimizer,
                                                                              accuracy_fn=accuracy_fn,
                                                                            )
  train_losses.append(train_loss); train_accuracies.append(train_acc)

  y_pred_valid, y_target_valid, valid_loss, valid_acc = test_step(data_loader=test_dataloader,
                                                                            model=model_2,
                                                                            loss_fn=loss_fn,
                                                                            accuracy_fn=accuracy_fn,
                                                                            scheduler=scheduler
                                                                            )
  valid_losses.append(valid_loss); valid_accuracies.append(valid_acc)

  torch.cuda.empty_cache()

  if valid_acc >= 99.9:
    EPOCHS = epoch+1
    break

In [ ]:
from helper_functions import accuracy_fn

p = True

while p:
  model_2 = KnotsModelCNN(output_shape=1
                        ).to(device)

  loss_fn = nn.MSELoss()
  optimizer = torch.optim.AdamW(params=model_2.parameters(),
                                lr=0.0003,
                                weight_decay=0.0001)
  scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,
                                                          mode='min',
                                                          factor=0.2,
                                                          patience=2,
                                                          threshold=0.001)

  train_losses, train_accuracies = [], []
  valid_losses, valid_accuracies = [], []

  y_pred_train, y_target_train = [], []
  y_pred_valid, y_target_valid = [], []

  EPOCHS = 20
  best_acc = 0
  best_fold = 0
  fold = 0
  SAVE_PATH = 'content/saved_models'

  try:
    os.mkdir(SAVE_PATH)
  except:
    pass

  for epoch in tqdm.notebook.tqdm(range(EPOCHS)):
    print(f'\nEpoch: {epoch+1}')
    print(f'LR: {scheduler.get_last_lr()}')

    y_pred_train, y_target_train, train_loss, train_acc = train_step(data_loader=train_dataloader,
                                                                                model=model_2,
                                                                                loss_fn=loss_fn,
                                                                                optimizer=optimizer,
                                                                                accuracy_fn=accuracy_fn,
                                                                              )
    train_losses.append(train_loss); train_accuracies.append(train_acc)

    y_pred_valid, y_target_valid, valid_loss, valid_acc = test_step(data_loader=test_dataloader,
                                                                              model=model_2,
                                                                              loss_fn=loss_fn,
                                                                              accuracy_fn=accuracy_fn,
                                                                              scheduler=scheduler
                                                                              )
    valid_losses.append(valid_loss); valid_accuracies.append(valid_acc)

    torch.cuda.empty_cache()

    if valid_acc >= 99.9:
      p = False
      EPOCHS = epoch+1
      break

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 10))
plt.title(f"Loss")

plt.plot(range(EPOCHS), train_losses, label="Train", linewidth=2)
plt.plot(range(EPOCHS), valid_losses, label="Validation", linewidth=2)

plt.legend()
plt.xlabel("Epoch")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 10))
plt.title("Accuracy")

plt.plot(range(EPOCHS), train_accuracies, label="Train", linewidth=2)
plt.plot(range(EPOCHS), valid_accuracies, label="Test", linewidth=2)

plt.legend()
plt.xlabel("Epoch")
plt.show()

### Plotting confusion matrixes

In [ ]:
import sklearn.metrics

In [ ]:
conf_matrix_train = sklearn.metrics.confusion_matrix(y_target_train, y_pred_train, labels=list(range(0, len(data.classes))))

conf_matrix_train

In [ ]:
conf_matrix_valid = sklearn.metrics.confusion_matrix(y_target_valid, y_pred_valid, labels=list(range(0, len(data.classes))))

conf_matrix_valid

In [ ]:
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(15, 5))

ax1.set_title("Train")
ax2.set_title("Test")

sklearn.metrics.ConfusionMatrixDisplay(confusion_matrix=conf_matrix_train).plot(ax=ax1)
sklearn.metrics.ConfusionMatrixDisplay(confusion_matrix=conf_matrix_valid).plot(ax=ax2)

### Plot weights

In [ ]:
model_2

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plot_weights([model_2.conv_1[0], model_2.conv_1[4], model_2.conv_1[8], model_2.conv_1[12], model_2.conv_1[16], model_2.classifier[3], model_2.classifier[7], model_2.classifier[9]])

In [ ]:
random_idx = torch.randint(0, len(train_features_batch), size=[1]).item()
img = train_features_batch[random_idx]
plt.imshow(img.squeeze(), cmap="gray")
plt.axis("Off")

In [ ]:
func = []
to_be_shown = (torch.nn.modules.linear.Linear, torch.nn.modules.Conv2d)
func_l = 0

for name, module in model_2.named_modules():
  l = len(name.split('.'))
  if l >= 2:
    func.append(module)
    if type(module) in to_be_shown:
      func_l += 1

func, func_l

In [ ]:
plot_weights_image(func, func_l, img)

### Evaluating and saving the model

In [ ]:
# Calculate model_1 results with device-agnostic code
model_2_results = eval_model(model=model_2,
                             data_loader=valid_dataloader,
                             loss_fn=loss_fn,
                             accuracy_fn=accuracy_fn,
                             device=device
                         )
model_2_results

In [ ]:
from pathlib import Path

model_2.load_state_dict(torch.load(Path("/content/CNN.pth"), weights_only=True))

In [ ]:
from pathlib import Path

# 1. Create models directory
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# 2. Create model save path
MODEL_NAME = "CNN.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

# 3. Save the model state dict
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model_2.state_dict(), # only saving the state_dict() only saves the models learned parameters
           f=MODEL_SAVE_PATH)

In [ ]:
# Instantiate a fresh instance of KnotsModelVanila
loaded_model_2 = KnotsModelCNN(hidden_units_conv=2,
                        hidden_units_fc=1024,
                        output_shape=1
                        ).to(device) # send model to GPU if it's available
next(loaded_model_2.parameters()).device # check model device

# Load model state dict
loaded_model_2.load_state_dict(torch.load(Path("models/CNN.pth")))

# Put model to target device (if your data is on GPU, model will have to be on GPU to make predictions)
loaded_model_2.to(device)

print(f"Loaded model:\n{loaded_model_2}")
print(f"Model on device:\n{next(loaded_model_2.parameters()).device}")

In [ ]:
model_2 = loaded_model_2

In [ ]:
model_2_loaded_results = eval_model(model=model_2,
                                    data_loader=test_dataloader,
                                    loss_fn=loss_fn,
                                    accuracy_fn=accuracy_fn,
                                    device=device
                                )
model_2_loaded_results

In [ ]:
from PIL import Image
import torchvision
from torchvision.transforms import v2

data_transform = v2.Compose([v2.Grayscale(num_output_channels=1),
                             v2.ToImage(),
                             v2.ToDtype(torch.float32, scale=True),
                             v2.functional.invert])

img_1 = data_transform(Image.open('/content/3_1.png')).unsqueeze(dim=0).to(device)
img_2 = data_transform(Image.open('/content/4_1.png')).unsqueeze(dim=0).to(device)

img = torch.cat([img_1, img_2], dim=0)

model_2(img)